# Librerias importadas

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.ensemble import BaggingClassifier
from sklearn.model_selection import train_test_split 
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_curve, auc, roc_auc_score, confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve, auc
from sklearn.metrics import make_scorer, mean_absolute_error, mean_squared_error, r2_score

### **Explicación del código y los pasos previos al modelo:**

**Carga y limpieza de datos:**
   - **Cargar el archivo Excel**: Utilizo `pandas.read_excel()` para cargar el conjunto de datos desde el archivo `datos_tarea25.xlsx`.
   - **Limpieza de las columnas**: El código elimina valores nulos en columnas como **`Levy`**, **`Mileage`**, y **`Engine volume`**.

         - Se limpian los valores nulos en la columna Levy '-' sustituyendolos por NaN e imputandole a esos valores la mediana (para que no haya                      missings). Luego la paso a numerica.
         Se eligió imputar los valores faltantes de la columna Levy con la mediana porque esta técnica es menos sensible a los outliers y es adecuada para datos numéricos con distribución sesgada, como puede ser el caso en los precios. El uso de la media podría haber sesgado los resultados si existieran valores extremos, esto no ocurre con la mediana.
     
         - Para Mileage se comprueba que contienen ' km' detras del valor numerico se limpia ese string y se transforma en numerica.
     
         - Lo mismo para Engine_volume, omitimos el valor 'Turbo' y convertimos a numerico
     
     
   - **Codificación One-Hot**: Se utiliza para las columnas categóricas (como **`Manufacturer`**, **`Category`**, **`Fuel type`**, etc.) Ademas para la columna **`Wheel`** se utiliza **`pd.get_dummies()`**, así las transformo en variables numéricas (columnas binarias) que los modelos de machine learning pueden procesar.
   - **Codificación de `Leather interior`**: Aquí codifico la columna **`Leather interior`** con valores binarios (1 para "Yes" y 0 para "No").
   - **Eliminación de duplicados**: El código elimina cualquier fila duplicada en el dataset usando **`drop_duplicates()`**, lo que evita que haya registros redundantes que puedan sesgar el modelo.

    Se utiliza OneHot Encoding para las variables categóricas porque no existe un orden natural en las categorías, como en el caso de FuelType (por ejemplo, 'Diesel', 'Gasolina', etc.). Porque este tipo de codificación es adecuado para modelos SVM que requieren datos numéricos y no pueden interpretar las categorías directamente.

In [ ]:
# Cargar el archivo Excel
archivo_tarea = r'C:\Users\MAICKEL\OneDrive\Documents\MASTER_UCM\MODULO 8 - P0 - MACHINE LEARNING (INMACULADA GUTI)\Documentación machine learning - Inmaculada Gutiérrez-20250306\Tarea\datos_tarea25.xlsx'
df = pd.read_excel(archivo_tarea)
print(df.head())

In [ ]:
# Limpieza de la columna Levy:
df['Levy'] = df['Levy'].replace('-', np.nan)
# Crear un imputador para reemplazar los NaN por la mediana
imputer = SimpleImputer(strategy='median')
df['Levy'] = imputer.fit_transform(df[['Levy']]) # Imputar la mediana en la columna 'Levy'
df['Levy'] = pd.to_numeric(df['Levy'], errors='coerce')

# Limpieza de la columna Mileage:
df['Mileage'] = df['Mileage'].str.replace(' km', '', regex=False)
df['Mileage'] = pd.to_numeric(df['Mileage'], errors='coerce')

# Verificar los cambios en Levy y Mileage:
print(df[['Levy', 'Mileage']].head())

# Limpiar la columna 'Engine volume', eliminando ' Turbo'
df['Engine volume'] = df['Engine volume'].str.replace(' Turbo', '', regex=False)  # Eliminar ' Turbo'
df['Engine volume'] = pd.to_numeric(df['Engine volume'], errors='coerce')  # Convertir a numérico

In [ ]:
df = pd.get_dummies(df, columns=['Wheel'], drop_first=True)

# Listar todas las columnas numéricas y categoricas que necesitan ser escaladas y tratadas respectivamente
numeric_columns = ['Price', 'Levy', 'Mileage', 'Engine volume', 'Airbags']
categorical_columns = ['Manufacturer', 'Category', 'Fuel type', 'Gear box type', 'Drive wheels']

encoder = OneHotEncoder(sparse_output=False) # Inicializar el codificador OneHotEncoder
encoded_features = encoder.fit_transform(df[categorical_columns])
encoded_df = pd.DataFrame(encoded_features, columns=encoder.get_feature_names_out(categorical_columns))
df = df.join(encoded_df).drop(columns=categorical_columns)

# Codificar 'Leather interior' con 1 para 'Yes' y 0 para 'No'
df['Leather interior'] = df['Leather interior'].map({'Yes': 1, 'No': 0})

In [ ]:
# Revisar duplicados: (filas completamente duplicadas en todas las columnas, se eliminan para evitar sobreajuste)
print("Duplicados:", df.duplicated().sum())
# Eliminar duplicados exactos
df = df.drop_duplicates()
# Verificar si todavía quedan duplicados
print("Duplicados después de eliminar:", df.duplicated().sum())

In [ ]:
# Revisar la variable Color para homogeneizar valores:
df['Color'] = df['Color'].str.strip().str.capitalize()
print(df['Color'].value_counts())
# Codificar la columna 'Color' en 0 (Black) y 1 (White)
df['Color'] = df['Color'].apply(lambda x: 1 if x == 'White' else 0)
print(df['Color'].value_counts())

# Crear mejor modelo SVM

**Separación en `train/test`:**
   - **División de datos**: Utilizas `train_test_split` para dividir el dataset en un conjunto de entrenamiento (70%) y uno de prueba (30%). Esta separación permite entrenar el modelo con una parte de los datos y evaluarlo con otra parte no vista durante el entrenamiento.
   
**Estandarización**:
   - La **estandarización** se realiza después de dividir los datos en **`train/test`** para evitar el **data leakage**. El **data leakage** ocurre cuando los datos de prueba influyen en el modelo de entrenamiento, lo que puede resultar en un rendimiento artificialmente alto y en un modelo que no generaliza bien.
   - El **`StandardScaler()`** es utilizado para que las características numéricas tengan media 0 y desviación estándar 1. Este paso es fundamental para algoritmos como **SVM**, ya que estos modelos son sensibles a las escalas de las características. Si las variables no están estandarizadas, algunas podrían dominar el cálculo de la distancia, lo que afecta el rendimiento del modelo.

In [ ]:
seed = 123     
X = df.drop(columns=['Color'])  
y = df['Color'] 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=seed)

scaler = StandardScaler()
X_train_tr = scaler.fit_transform(X_train)
X_test_tr = scaler.transform(X_test)

**Modelo SVM y búsqueda de hiperparámetros:**
   - **SVM**: Se crea un clasificador **SVM** (Support Vector Machine) con el kernel **`rbf`**, **`poly`** y **`linear`**.
   - **`GridSearchCV`**: El **`GridSearchCV`** se usa para buscar el mejor conjunto de parámetros **`C`**, **`kernel`**, y **`gamma`**. Esto se realiza mediante validación cruzada (3 pliegues en este caso). El modelo es evaluado con diferentes combinaciones de estos parámetros, y el conjunto que da el mejor rendimiento en términos de **accuracy** es seleccionado.
   - Voy probando distintos parametros hasta quedarme con el mejor modelo en terminos de precisión y dejando el siguiente param_grid:

In [ ]:
param_grid = {
    'C': [0.1, 1],  # Menos valores de C
    'kernel': ['linear', 'rbf'],  # Mantén los dos kernels
    'gamma': ['scale', 'auto']  # Solo probamos 'scale' para gamma
}

# Crear el clasificador SVM
svm = SVC(probability=True, random_state=seed)
# Configurar GridSearchCV con validación cruzada
grid_search = GridSearchCV(estimator=svm, param_grid=param_grid, scoring='accuracy', cv=3, verbose=1, n_jobs=-1)
grid_search.fit(X_train_tr, y_train) # Ajustar el modelo SVM con el conjunto de entrenamiento

# Obtener los mejores parámetros y el mejor modelo
best_params = grid_search.best_params_
best_model = grid_search.best_estimator_
# Mostrar los mejores parámetros encontrados
print(f"Mejores parámetros encontrados: {best_params}")
print(f"Mejor estimador encontrado: {best_model}")

In [ ]:
y_pred = best_model.predict(X_train_tr)
y_prob = best_model.predict_proba(X_test_tr)[:, 1] 
roc_auc = roc_auc_score(y_test, y_prob)
print(f"AUC del mejor modelo: {roc_auc:.2f}")

fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc_val = auc(fpr, tpr)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {roc_auc_val:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC para el mejor modelo SVM')
plt.legend(loc='lower right')
plt.show()

# Predicciones con el mejor modelo encontrado
grid_predictions = grid_search.predict(X_test_tr)

cm = confusion_matrix(y_test, grid_predictions)
print("Matriz de Confusión:")
print(cm)

accuracy = (cm[0, 0] + cm[1, 1]) / (cm.sum())
print(f'Accuracy con el mejor modelo: {accuracy * 100:.2f}%')

### **Explicación de los resultados obtenidos modelo SVM:**

1. **Mejores parámetros**:
   El modelo **SVM** ha encontrado que el **mejor conjunto de parámetros** es:
   - **C = 0.1**
   - **kernel = rbf(radial Basis Function)**
   - **gamma = scale**

   El kernel **`rbf`** es una opción común para **SVM** en clasificación, ya que es capaz de manejar relaciones no lineales entre las características.

2. **Precisión del modelo (Accuracy)**:
   La precisión del modelo es **64%**. Esto significa que el modelo ha acertado **64% de las veces** en las predicciones sobre los datos de prueba. Esta precisión es moderada, lo que indica que el modelo tiene un rendimiento aceptable, pero hay margen para mejorar.

   - **Accuracy** se calcula como:
     \[
     \text{Accuracy} = \frac{\text{TP} + \text{TN}}{\text{TP} + \text{TN} + \text{FP} + \text{FN}}
     \]
     donde:
     - **TP** = Verdaderos positivos
     - **TN** = Verdaderos negativos
     - **FP** = Falsos positivos
     - **FN** = Falsos negativos

3. **AUC (Área bajo la curva ROC)**:
   El valor **AUC** es **0.67**, lo que sugiere que el modelo tiene una capacidad moderada para discriminar entre las dos clases (0 y 1). Un **AUC de 1** significaría un modelo perfecto, mientras que **0.5** es aleatorio (sin capacidad para clasificar correctamente).

4. **Matriz de confusión**:


### **Cálculo de la precisión (accuracy)**:
La **precisión** (accuracy) es la fracción de predicciones correctas con respecto a todas las predicciones:

[\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN} = \frac{250 + 289}{250 + 289 + 141 + 162} = \frac{539}{842} \approx 0.64 \quad (\text{64% de precisión})
\]

### **Resumen MC**:

- **TP** = 250
- **TN** = 289
- **FP** = 141
- **FN** = 162

- **Precisión (Accuracy)** = **64%**

### **Curva ROC**:
La **curva ROC** visualiza la tasa de verdaderos positivos (**TPR**) frente a la tasa de falsos positivos (**FPR**) para diferentes umbrales de decisión. Un AUC de **0.67** (como lo ves en la gráfica) indica que el modelo es capaz de clasificar correctamente más de la mitad de las veces, pero tiene espacio para mejorar. El modelo está mejor que una clasificación aleatoria (que estaría en la línea diagonal).

### **En resumen:**
- **Precisión**: 64% muestra que el modelo tiene una capacidad razonable para clasificar correctamente, pero aún hay margen para mejorar.
- **AUC**: 0.67 muestra que el modelo tiene una discriminación moderada.

# CREAR MODELO BAGGING A PARTIR DEL MEJOR MODELO

In [ ]:
# Crear el clasificador de Bagging con el mejor modelo SVM
bagging_model = BaggingClassifier(estimator=best_model,  
                                 n_estimators=50,  # Número de modelos SVM
                                 random_state=seed)

bagging_model.fit(X_train_tr, y_train)

In [ ]:
# Predicciones del modelo de Bagging
y_pred_bagging = bagging_model.predict(X_test_tr)
y_prob_bagging = bagging_model.predict_proba(X_test_tr)[:, 1]  # Probabilidades para AUC

# Calcular la precisión (Accuracy)
accuracy_bagging = accuracy_score(y_test, y_pred_bagging)
print(f"Precisión del modelo Bagging: {accuracy_bagging * 100:.2f}%")

In [ ]:
# Calcular AUC
roc_auc_bagging = roc_auc_score(y_test, y_prob_bagging)
print(f"AUC del modelo Bagging: {roc_auc_bagging:.2f}")

print(f"Precisión del SVM original: {accuracy * 100:.2f}%")
print(f"AUC del SVM original: {roc_auc:.2f}")
print(f"Precisión del Bagging: {accuracy_bagging * 100:.2f}%")
print(f"AUC del Bagging: {roc_auc_bagging:.2f}")

fpr_bagging, tpr_bagging, thresholds_bagging = roc_curve(y_test, y_prob_bagging)
roc_auc_val_bagging = auc(fpr_bagging, tpr_bagging)

plt.figure(figsize=(8, 6))
plt.plot(fpr_bagging, tpr_bagging, color='darkorange', lw=2, label=f'Bagging ROC curve (AUC = {roc_auc_val_bagging:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC para el modelo de Bagging')
plt.legend(loc='lower right')
plt.show()

Parece que tanto el modelo **SVM** como el modelo **Bagging** tienen resultados muy similares, con una **precisión** del 64.37% y un **AUC** de 0.67. Esto indica que, aunque ambos modelos están ofreciendo resultados relativamente buenos, la técnica de **bagging** no ha proporcionado una mejora significativa en comparación con el modelo SVM original.

# MODELO STACKING

In [ ]:
# MODELO STACKING 1
base_learners = [
    ('svm', SVC(probability=True, kernel='linear', random_state=seed)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=seed)),
    ('knn', KNeighborsClassifier())
]

meta_model = LogisticRegression(random_state=seed)

stacking_model = StackingClassifier(estimators=base_learners, final_estimator=meta_model)
stacking_model.fit(X_train_tr, y_train)

In [ ]:
# Predicciones del modelo de Stacking
y_pred_stacking = stacking_model.predict(X_test_tr)
y_prob_stacking = stacking_model.predict_proba(X_test_tr)[:, 1]  # Probabilidades para AUC

# Calcular la precisión (Accuracy)
accuracy_stacking = accuracy_score(y_test, y_pred_stacking)
print(f"Precisión del modelo de Stacking: {accuracy_stacking * 100:.2f}%")

In [ ]:
# Calcular AUC
roc_auc_stacking = roc_auc_score(y_test, y_prob_stacking)
print(f"AUC del modelo de Stacking: {roc_auc_stacking:.2f}")

# GRAFICAR CURVA ROC
fpr_stacking, tpr_stacking, thresholds_stacking = roc_curve(y_test, y_prob_stacking)
roc_auc_val_stacking = auc(fpr_stacking, tpr_stacking)

plt.figure(figsize=(8, 6))
plt.plot(fpr_stacking, tpr_stacking, color='darkorange', lw=2, label=f'Stacking ROC curve (AUC = {roc_auc_val_stacking:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC para el modelo de Stacking')
plt.legend(loc='lower right')
plt.show()

El modelo de **Stacking** ha tenido una **precisión** de **61.28%** y un **AUC** de **0.67**, lo que lo hace un poco menos preciso que el modelo **SVM** (que tiene una precisión de **64.01%**). Sin embargo, el **AUC** es el mismo en ambos casos, lo que indica que, a pesar de la disminución en precisión, ambos modelos están logrando una capacidad similar de clasificación en términos de la **curva ROC**.

### Posibles Explicaciones:
1. **Combinación de modelos base**: En este caso, el stacking está combinando el SVM, Random Forest y KNN. Es posible que la combinación de estos modelos no esté resultando en una mejora significativa. Puede que algunos modelos base no estén aportando valor al modelo final.
   
2. **Ajustes en los modelos base o meta**: La **regresión logística** utilizada como modelo meta podría no ser la mejor opción para combinar las predicciones de los modelos. Se podría intentar con otros modelos más complejos como **XGBoost** o un **Random Forest** para ver si mejora.

In [ ]:
# MODELO STACKING 2
base_learners = [
    ('gb', GradientBoostingClassifier(n_estimators=100, random_state=seed)),
    ('rf', RandomForestClassifier(n_estimators=100, random_state=seed)),
    ('knn', KNeighborsClassifier())
]

# Definir el clasificador meta
meta_model = RandomForestClassifier(n_estimators=100, random_state=seed)

# Crear el modelo de Stacking
stacking_model2 = StackingClassifier(estimators=base_learners, final_estimator=meta_model)
stacking_model2.fit(X_train_tr, y_train)

# Predicciones del modelo de Stacking
y_pred_stacking = stacking_model2.predict(X_test_tr)
y_prob_stacking = stacking_model2.predict_proba(X_test_tr)[:, 1]  # Probabilidades para AUC

# Calcular la precisión (Accuracy)
accuracy_stacking2 = accuracy_score(y_test, y_pred_stacking)
print(f"Precisión del modelo de Stacking 2: {accuracy_stacking * 100:.2f}%")

# Calcular AUC
roc_auc_stacking2 = roc_auc_score(y_test, y_prob_stacking)
print(f"AUC del modelo de Stacking 2: {roc_auc_stacking:.2f}")

# GRAFICAR CURVA ROC
fpr_stacking, tpr_stacking, thresholds_stacking = roc_curve(y_test, y_prob_stacking)
roc_auc_val_stacking = auc(fpr_stacking, tpr_stacking)

plt.figure(figsize=(8, 6))
plt.plot(fpr_stacking, tpr_stacking, color='darkorange', lw=2, label=f'Stacking ROC curve (AUC = {roc_auc_val_stacking:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Curva ROC para el modelo de Stacking 2')
plt.legend(loc='lower right')
plt.show()

In [ ]:
# Lista de modelos ya definidos
models = [
    ('SVM', best_model), 
    ('Bagging', bagging_model), 
    ('Stacking', stacking_model),
    ('Stacking 2', stacking_model2)
]

# Listas para almacenar las precisiones de cada modelo
precisiones = []

# Aplicar validación cruzada a cada modelo
for model_name, model in models:
    cv_scores = cross_val_score(model, X_train_tr, y_train, cv=5, scoring='accuracy')  # 5-fold cross-validation
    precisiones.append(cv_scores)

# Crear el boxplot
plt.figure(figsize=(8, 6))
sns.boxplot(data=precisiones)

# Etiquetas para los modelos
modelos = [model[0] for model in models]

# Personalizar gráfico
plt.xticks(ticks=range(len(modelos)), labels=modelos)
plt.title('Comparación de Modelos en términos de Precisión (Validación Cruzada)')
plt.xlabel('Modelos')
plt.ylabel('Precisión (%)')
plt.show()

In [ ]:
for model_name, model in models:
    model.fit(X_train_tr, y_train)
    y_pred = model.predict(X_train_tr)
    y_test_pred = model.predict(X_test)

    train_accuracy = accuracy_score(y_train, y_pred)
    test_accuracy = accuracy_score(y_test, y_test_pred)
    
    print(f"\nModelo: {model_name}")
    print(f"Precisión en Train: {train_accuracy:.4f}")
    print(f"Precisión en Test: {test_accuracy:.4f}")
    print("-" * 50)
    
    cm = confusion_matrix(y_train, y_pred)
    plt.figure(figsize=(2, 2))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
    plt.title(f'Matriz de Confusión - {model_name}')
    plt.show()

### Análisis de los resultados basados en la matriz de confusión:

1. **SVM**:
   - **Precisión en Train**: 69.60%
   - **Precisión en Test**: 51.07%
   - **Matriz de Confusión**: El modelo muestra una **buena capacidad para clasificar correctamente la clase negativa** (716 verdaderos negativos) y un número razonable de verdaderos positivos (651). Sin embargo, la **precisión en test** es relativamente baja, lo que sugiere que el modelo puede estar sobreajustado.

2. **Bagging**:
   - **Precisión en Train**: 70.26%
   - **Precisión en Test**: 51.07%
   - **Matriz de Confusión**: Similar al SVM, el modelo **Bagging** muestra un rendimiento en **train** bastante bueno, pero la **precisión en test** no es superior. La matriz muestra una distribución de clases en las predicciones, pero la precisión general es similar al SVM.

3. **Stacking**:
   - **Precisión en Train**: 99.69%
   - **Precisión en Test**: 53.68%
   - **Matriz de Confusión**: El modelo de **Stacking** tiene un **rendimiento extremadamente bueno en train**, lo que indica que puede estar sobreajustando los datos. Sin embargo, su precisión en test es relativamente baja (53.68%), y la matriz muestra que tiene muchos **falsos negativos** y **falsos positivos**, lo que sugiere que el modelo no está generalizando bien a los datos de prueba.

4. **Stacking 2**:
   - **Precisión en Train**: 75.66%
   - **Precisión en Test**: 53.68%
   - **Matriz de Confusión**: Similar al **Stacking**, el modelo **Stacking 2** muestra una precisión decente en el conjunto de entrenamiento, pero en el conjunto de prueba la precisión sigue siendo baja. La matriz también muestra ciertos problemas con la clasificación de las clases.

### Conclusión:

- **SVM** parece ser el modelo más equilibrado en términos de rendimiento, aunque con una **precisión en test** de 51.07%. A pesar de que no es el modelo más preciso en cuanto a la precisión de test, tiene un buen rendimiento en la validación cruzada y no muestra signos de sobreajuste tan evidentes como el **Stacking**. El modelo SVM tiene la ventaja de ser un modelo relativamente simple y eficiente, con un rendimiento que no varía drásticamente entre entrenamiento y prueba.
  
- **Bagging** muestra una precisión similar a la del SVM, con una variabilidad igualmente baja, lo que también lo hace un candidato sólido.

- **Stacking** y **Stacking 2** parecen estar **sobreajustando** los datos de entrenamiento, ya que muestran una **precisión alta en train**, pero el rendimiento en test es deficiente y no mejora mucho en comparación con el **SVM**.

### Elección Final:
**SVM** sigue siendo la opción más adecuada por su **simplicidad**, su **bajo riesgo de sobreajuste**, y teniendo un rendimiento similar a modelos mas complejos pese a que la precision en test de todos los modelos es muy baja, siendo casi **aleatoria** sus predicciones. Aunque el **Bagging** es competitivo, **SVM** es más eficiente y fácil de interpretar.